# 02 · Build all comma2k19 processed data

Resume-safe: existing video + metadata pairs are skipped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

# ============================================================
# Repository
# ============================================================
REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

# ============================================================
# Paths
# ============================================================
DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"

COMMA_ROOT = DATA_ROOT / "comma2k19"
RAW_ROOT = COMMA_ROOT / "raw"
PROCESSED_ROOT = COMMA_ROOT / "processed" / "v1"

MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
# Install project from the existing pyproject.toml
# --no-deps keeps Colab's/DACON's binary stack intact.
# ============================================================
subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q", "--no-deps", "-e", str(REPO),
    ],
    check=True,
)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository     :", REPO)
print("Branch         :", BRANCH)
print("Commit         :", commit)
print("RAW_ROOT       :", RAW_ROOT)
print("PROCESSED_ROOT :", PROCESSED_ROOT)

In [ ]:
import pandas as pd
from blackbox_detection.stage3.comma2k19 import (
    find_archives,
    PrepareConfig,
    prepare_archive,
)
from blackbox_detection.stage3.manifest import build_segment_manifest

archives = find_archives(RAW_ROOT)
assert archives, f"No archives found under {RAW_ROOT}"

print("archives:", len(archives))
for p in archives:
    print(" -", p.name)

cfg = PrepareConfig(processed_root=PROCESSED_ROOT, overwrite=False)

all_reports = []
for i, archive in enumerate(archives, 1):
    print(f"\n[{i}/{len(archives)}] {archive.name}")
    rep = prepare_archive(archive, cfg)
    all_reports.append(rep)

    if "error" in rep.columns:
        bad = rep[rep["error"].notna()]
        if len(bad):
            print(f"errors in {archive.name}: {len(bad)}")
            display(bad)

report = pd.concat(all_reports, ignore_index=True)
report.to_csv(PROCESSED_ROOT / "prepare_report.csv", index=False)

manifest = build_segment_manifest(PROCESSED_ROOT)
manifest.to_csv(PROCESSED_ROOT / "manifest.csv", index=False)

print("\nsegments:", len(manifest))
print("frames  :", int(manifest.num_frames.sum()))
display(manifest.head())

In [ ]:
import shutil

usage = shutil.disk_usage("/content/drive/MyDrive")

video_gib = sum(
    p.stat().st_size
    for p in (PROCESSED_ROOT / "videos").rglob("*.mp4")
) / 2**30

metadata_gib = sum(
    p.stat().st_size
    for p in (PROCESSED_ROOT / "metadata").rglob("*.npz")
) / 2**30

print(f"Drive free          : {usage.free / 2**30:.1f} GiB")
print(f"Processed video GiB : {video_gib:.2f}")
print(f"Metadata GiB        : {metadata_gib:.2f}")